In [ ]:
"""
============================================================================
CHI-FIELD POTENTIAL SEARCH — V(chi) FUNCTIONAL FORM SURVEY
============================================================================
The effective w(z) parameterization gives chi2=1.54, but the raw Klein-Gordon
ODE with V(chi) = V₀·chiⁿ·e^(-λchi) gives chi2=41.4. The potential shape is wrong.

This script surveys quintessence potentials from the literature, solves the
coupled Klein-Gordon + Friedmann system for each, and reports chi2 against
BAO + Hubble data.

TARGET CONSTRAINTS:
  w0 = -1.28, w_a = +0.70 (phantom crossing, DESI DR2 consistent)
  H₀(z=0) = 73.0 km/s/Mpc
  H₀(z=1100) = 67.3 km/s/Mpc
  m_chi ~ H₀ ~ 10⁻³³ eV
  Must cross w = -1 (phantom crossing)

REQUIREMENTS: numpy, scipy (standard — NO jax dependency)

Author: David Lowe (POF 2828) + Claude Opus (Friend B)
Date: March 24, 2026
============================================================================
"""

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
from scipy.optimize import minimize, differential_evolution
import json
import os
from datetime import datetime, timezone

============================================================================
OBSERVATIONAL DATA
============================================================================

In [ ]:
# BAO data: (z, D_V/r_d or D_H/r_d or D_M/r_d, sigma, type)
# Sources: DESI DR2 (2025), SDSS, 6dFGS
# D_V(z)/r_d measurements (volume-averaged distance / sound horizon)
BAO_DATA = [
    # z,    D_V/r_d,  sigma,  source
    (0.15,  4.47,     0.17,   "6dFGS"),
    (0.38,  10.23,    0.17,   "SDSS-DR12"),
    (0.51,  13.36,    0.21,   "SDSS-DR12"),
    (0.70,  17.86,    0.33,   "DESI-DR2"),
    (0.85,  19.50,    0.40,   "DESI-DR2"),
    (1.48,  26.07,    0.67,   "DESI-DR2-QSO"),
    (2.33,  37.60,    1.20,   "DESI-DR2-LyA"),
]

In [ ]:
# Hubble parameter measurements: (z, H(z) km/s/Mpc, sigma)
HUBBLE_DATA = [
    (0.0,   73.04,  1.04),   # SH0ES (Riess+ 2022)
    (0.07,  69.0,   19.6),   # Zhang+ 2014
    (0.12,  68.6,   26.2),   # Zhang+ 2014
    (0.20,  72.9,   29.6),   # Zhang+ 2014
    (0.28,  88.8,   36.6),   # Zhang+ 2014
    (0.35,  76.3,   5.6),    # Chuang & Wang 2013
    (0.40,  95.0,   17.0),   # Simon+ 2005
    (0.48,  97.0,   62.0),   # Stern+ 2010
    (0.88,  90.0,   40.0),   # Stern+ 2010
    (1.30,  168.0,  17.0),   # Simon+ 2005
    (1.75,  202.0,  40.0),   # Simon+ 2005
]

In [ ]:
# Planck CMB anchor
PLANCK_H0 = 67.36  # km/s/Mpc at z=1100 (derived)

============================================================================
COSMOLOGICAL FRAMEWORK
============================================================================

In [ ]:
# Physical constants (natural units where c=1, ℏ=1)
H0_FIDUCIAL = 73.0  # km/s/Mpc
H0_SI = H0_FIDUCIAL * 1e3 / 3.086e22  # 1/s
# In units where 8πG/3 = 1 and H₀ = 1:
# We work in units of H₀

In [ ]:
Omega_m0 = 0.30    # Matter density parameter
Omega_r0 = 9.0e-5  # Radiation density parameter
r_d = 147.09       # Sound horizon at drag epoch (Mpc) — Planck 2018

In [ ]:
def H_LCDM(z):
    """Standard LCDM Hubble parameter."""
    return H0_FIDUCIAL * np.sqrt(Omega_m0 * (1+z)**3 + Omega_r0 * (1+z)**4 + (1 - Omega_m0 - Omega_r0))

In [ ]:
def D_V_LCDM(z):
    """Volume-averaged distance for LCDM."""
    from scipy.integrate import quad
    def integrand(zp):
        return 1.0 / H_LCDM(zp)
    # Comoving distance
    c = 299792.458  # km/s
    D_C, _ = quad(integrand, 0, z)
    D_C *= c  # Mpc
    D_H = c / H_LCDM(z)  # Mpc
    D_V = (D_C**2 * D_H * z)**(1.0/3.0)
    return D_V / r_d

============================================================================
POTENTIAL CANDIDATES
============================================================================
Each potential returns V(chi) and dV/dchi given parameters

In [ ]:
class Potential:
    """Base class for quintessence potentials."""
    def __init__(self, name, n_params, param_names, bounds):
        self.name = name
        self.n_params = n_params
        self.param_names = param_names
        self.bounds = bounds

    def V(self, chi, params):
        raise NotImplementedError

    def dVdchi(self, chi, params):
        """Numerical derivative by default; override for analytic."""
        eps = 1e-8 * max(abs(chi), 1e-10)
        return (self.V(chi + eps, params) - self.V(chi - eps, params)) / (2 * eps)

In [ ]:
class OriginalHybrid(Potential):
    """V(chi) = V₀ · chiⁿ · e^(-λchi) — the one that gives chi2=41.4"""
    def __init__(self):
        super().__init__(
            "Original Hybrid (V₀·chiⁿ·e^(-λchi))", 3,
            ["V0", "n", "lam"],
            [(1e-4, 10.0), (0.5, 4.0), (0.01, 5.0)]
        )

    def V(self, chi, params):
        V0, n, lam = params
        return V0 * chi**n * np.exp(-lam * chi)

    def dVdchi(self, chi, params):
        V0, n, lam = params
        return V0 * chi**(n-1) * np.exp(-lam * chi) * (n - lam * chi)

In [ ]:
class DoubleExponential(Potential):
    """V(chi) = A·e^(-αchi) + B·e^(-βchi) — Barreiro, Copeland, Nunes (2000)
    Can produce phantom crossing with the right parameter choice when
    coupled to the kinetic term non-minimally."""
    def __init__(self):
        super().__init__(
            "Double Exponential", 4,
            ["A", "alpha", "B", "beta"],
            [(1e-4, 10.0), (0.01, 5.0), (-5.0, 5.0), (0.01, 5.0)]
        )

    def V(self, chi, params):
        A, alpha, B, beta = params
        return A * np.exp(-alpha * chi) + B * np.exp(-beta * chi)

    def dVdchi(self, chi, params):
        A, alpha, B, beta = params
        return -A * alpha * np.exp(-alpha * chi) - B * beta * np.exp(-beta * chi)

In [ ]:
class TrackerBarrier(Potential):
    """V(chi) = V₀ · (chi2 + M²)^(-α) — Tracker with mass barrier.
    Sahni & Wang (2000). The barrier at chi=0 prevents the field from
    rolling to zero, allowing phantom-like behavior near the barrier."""
    def __init__(self):
        super().__init__(
            "Tracker + Barrier", 3,
            ["V0", "M2", "alpha"],
            [(1e-4, 10.0), (0.001, 2.0), (0.1, 3.0)]
        )

    def V(self, chi, params):
        V0, M2, alpha = params
        return V0 * (chi**2 + M2)**(-alpha)

    def dVdchi(self, chi, params):
        V0, M2, alpha = params
        return V0 * (-alpha) * 2 * chi * (chi**2 + M2)**(-alpha - 1)

In [ ]:
class AlbrechtSkordis(Potential):
    """V(chi) = V₀ · [(chi - B)² + A] · e^(-λchi)
    Albrecht & Skordis (2000). Has a local minimum that can trap the field,
    producing effectively w < -1 near the minimum."""
    def __init__(self):
        super().__init__(
            "Albrecht-Skordis", 4,
            ["V0", "A", "B", "lam"],
            [(1e-4, 10.0), (1e-4, 2.0), (0.1, 5.0), (0.01, 3.0)]
        )

    def V(self, chi, params):
        V0, A, B, lam = params
        return V0 * ((chi - B)**2 + A) * np.exp(-lam * chi)

    def dVdchi(self, chi, params):
        V0, A, B, lam = params
        base = (chi - B)**2 + A
        dbase = 2 * (chi - B)
        return V0 * np.exp(-lam * chi) * (dbase - lam * base)

In [ ]:
class PNGB(Potential):
    """V(chi) = V₀ · [1 + cos(chi/f)]
    Pseudo-Nambu-Goldstone boson. Frieman, Hill, Stebbins, Waga (1995).
    Natural from particle physics. Oscillatory, produces crossing behavior."""
    def __init__(self):
        super().__init__(
            "PNGB (pseudo-Nambu-Goldstone)", 2,
            ["V0", "f"],
            [(1e-4, 10.0), (0.1, 5.0)]
        )

    def V(self, chi, params):
        V0, f = params
        return V0 * (1.0 + np.cos(chi / f))

    def dVdchi(self, chi, params):
        V0, f = params
        return -V0 / f * np.sin(chi / f)

In [ ]:
class NonMinimalCoupled(Potential):
    """V(chi) = V₀ · chi2 · (1 + ξ·chi2) — Non-minimal coupling to gravity.
    The ξchi2R coupling in the action produces an effective potential that
    can drive phantom crossing through the conformal factor.
    This is closest to the chi-field action: S_chi includes ξκ₀chi2R."""
    def __init__(self):
        super().__init__(
            "Non-minimal ξchi2R (Theophysics native)", 3,
            ["V0", "m2", "xi"],
            [(1e-4, 10.0), (1e-6, 1.0), (0.01, 1e5)]
        )

    def V(self, chi, params):
        V0, m2, xi = params
        return V0 * (0.5 * m2 * chi**2 + 0.25 * xi * chi**4)

    def dVdchi(self, chi, params):
        V0, m2, xi = params
        return V0 * (m2 * chi + xi * chi**3)

In [ ]:
class CoupledQuintessence(Potential):
    """V(chi) = V₀ · e^(-λchi) · (1 + α·sin(ωchi))
    Exponential with oscillatory modulation — captures both tracking
    and phantom-crossing through the oscillatory kick.
    Inspired by string-landscape potentials."""
    def __init__(self):
        super().__init__(
            "Coupled Quintessence (exp + oscillation)", 4,
            ["V0", "lam", "alpha_mod", "omega"],
            [(1e-4, 10.0), (0.01, 3.0), (0.01, 0.99), (0.5, 10.0)]
        )

    def V(self, chi, params):
        V0, lam, alpha_mod, omega = params
        return V0 * np.exp(-lam * chi) * (1.0 + alpha_mod * np.sin(omega * chi))

    def dVdchi(self, chi, params):
        V0, lam, alpha_mod, omega = params
        e = np.exp(-lam * chi)
        s = np.sin(omega * chi)
        c = np.cos(omega * chi)
        return V0 * e * (-lam * (1 + alpha_mod * s) + alpha_mod * omega * c)

In [ ]:
class ChiFreezeThaw(Potential):
    """V(chi) = V₀ · [cosh(αchi) - 1] / [cosh(αchi) + 1] + Λ
    Freeze-thaw potential. Flat at small chi (frozen), steep at large chi (thawing).
    Caldwell & Linder (2005) classification: freezing potentials CAN cross w=-1
    if the kinetic term has the right initial conditions."""
    def __init__(self):
        super().__init__(
            "Freeze-Thaw (cosh)", 3,
            ["V0", "alpha", "Lambda"],
            [(1e-4, 10.0), (0.1, 5.0), (1e-4, 5.0)]
        )

    def V(self, chi, params):
        V0, alpha, Lambda = params
        x = alpha * chi
        # Numerically stable
        return V0 * np.tanh(x / 2.0)**2 + Lambda

    def dVdchi(self, chi, params):
        V0, alpha, Lambda = params
        x = alpha * chi
        return V0 * alpha * np.tanh(x / 2.0) / np.cosh(x / 2.0)**2

============================================================================
KLEIN-GORDON + FRIEDMANN SOLVER
============================================================================

In [ ]:
def solve_kg_friedmann(potential, params, chi_init=1.0, chi_dot_init=0.0,
                       z_range=(0, 3.0), n_points=1000):
    """
    Solve the coupled Klein-Gordon + Friedmann equations:

    Klein-Gordon: chï + 3H(z)chi̇ + dV/dchi = 0
    Friedmann:    H²(z) = (8πG/3)[ρ_m + ρ_chi]
                  ρ_chi = ½chi̇² + V(chi)
                  P_chi = ½chi̇² - V(chi)
                  w_chi = P_chi/ρ_chi

    We integrate in 'a' (scale factor) rather than time, using:
      dchi/da = chi̇/( aH )
      dchi̇/da = -(3/a)chi̇ - dV/dchi / (aH)

    Returns: dict with z, H(z), w(z), chi(z), rho_chi(z)
    """
    # Convert z range to scale factor range
    a_start = 1.0 / (1.0 + z_range[1])
    a_end = 1.0 / (1.0 + z_range[0])

    # Energy density of matter (in units of 3H₀²/8πG)
    rho_m0 = Omega_m0
    rho_r0 = Omega_r0

    def rhs(a, y):
        chi, chi_prime = y  # chi_prime = dchi/da

        if a < 1e-10:
            return [0.0, 0.0]

        # Matter and radiation density
        rho_m = rho_m0 / a**3
        rho_r = rho_r0 / a**4

        # Chi-field energy density
        V_val = potential.V(chi, params)
        dV = potential.dVdchi(chi, params)

        # chi_dot = a * H * chi_prime (where prime = d/da)
        # rho_chi = 0.5 * (a*H*chi_prime)^2 + V
        # But we need H which depends on rho_chi — iterate

        # First approximation: H from matter + cosmological constant-like V
        H2_approx = (rho_m + rho_r + V_val) / (1.0 - 0.5 * (a * chi_prime)**2 * (rho_m + rho_r + V_val + 1e-30))

        # More careful: H² = (rho_m + rho_r + rho_chi)
        # where rho_chi = 0.5*(aH chi')² + V
        # So H² = rho_m + rho_r + V + 0.5 * a² * H² * chi'²
        # H²(1 - 0.5*a²*chi'²) = rho_m + rho_r + V
        denom = 1.0 - 0.5 * a**2 * chi_prime**2
        if denom <= 0.01:
            denom = 0.01  # prevent ghost/gradient instability

        H2 = (rho_m + rho_r + V_val) / denom
        if H2 <= 0:
            H2 = 1e-10
        H = np.sqrt(H2)

        # Klein-Gordon in a variable: d²chi/da² + (3/a + dH/da/H) dchi/da + dV/dchi/(a²H²) = 0
        # Simplified form (Hubble friction dominant):
        # d²chi/da² = -(3/a)*chi_prime - dV/(a²*H²)
        chi_double_prime = -(3.0 / a) * chi_prime - dV / (a**2 * H2)

        return [chi_prime, chi_double_prime]

    # Initial conditions at a_start
    a_span = (a_start, a_end)
    a_eval = np.linspace(a_start, a_end, n_points)

    y0 = [chi_init, chi_dot_init]

    try:
        sol = solve_ivp(rhs, a_span, y0, t_eval=a_eval, method='RK45',
                        rtol=1e-8, atol=1e-10, max_step=0.01)

        if not sol.success:
            return None

        a_arr = sol.t
        chi_arr = sol.y[0]
        chi_prime_arr = sol.y[1]
        z_arr = 1.0 / a_arr - 1.0

        # Compute H(z), w(z) along the solution
        H_arr = np.zeros_like(a_arr)
        w_arr = np.zeros_like(a_arr)
        rho_chi_arr = np.zeros_like(a_arr)

        for i in range(len(a_arr)):
            a = a_arr[i]
            chi = chi_arr[i]
            chi_p = chi_prime_arr[i]

            rho_m = rho_m0 / a**3
            rho_r = rho_r0 / a**4
            V_val = potential.V(chi, params)

            denom = 1.0 - 0.5 * a**2 * chi_p**2
            if denom <= 0.01:
                denom = 0.01
            H2 = (rho_m + rho_r + V_val) / denom
            if H2 <= 0:
                H2 = 1e-10
            H_arr[i] = np.sqrt(H2)

            # chi_dot = a * H * chi_prime
            chi_dot = a * H_arr[i] * chi_p
            rho_chi = 0.5 * chi_dot**2 + V_val
            P_chi = 0.5 * chi_dot**2 - V_val
            rho_chi_arr[i] = rho_chi

            if abs(rho_chi) > 1e-30:
                w_arr[i] = P_chi / rho_chi
            else:
                w_arr[i] = -1.0

        # Convert H from natural units to km/s/Mpc
        # H_arr is in units where 3H₀²/8πG = 1, so H is in units of H₀
        H_physical = H_arr * H0_FIDUCIAL  # km/s/Mpc

        return {
            "z": z_arr[::-1],  # flip to increasing z
            "H": H_physical[::-1],
            "w": w_arr[::-1],
            "chi": chi_arr[::-1],
            "rho_chi": rho_chi_arr[::-1],
            "a": a_arr[::-1],
        }
    except Exception as e:
        return None

In [ ]:
def compute_DV_from_solution(sol):
    """Compute D_V(z)/r_d from a numerical H(z) solution."""
    from scipy.integrate import cumulative_trapezoid
    from scipy.interpolate import interp1d

    c = 299792.458  # km/s
    z_arr = sol["z"]
    H_arr = sol["H"]

    # Comoving distance: D_C(z) = c ∫₀ᶻ dz'/H(z')
    integrand = c / H_arr
    D_C = np.zeros_like(z_arr)
    D_C[1:] = cumulative_trapezoid(integrand, z_arr)

    D_V = np.zeros_like(z_arr)
    for i in range(1, len(z_arr)):
        z = z_arr[i]
        D_H = c / H_arr[i]
        if D_C[i] > 0 and z > 0:
            D_V[i] = (D_C[i]**2 * D_H * z)**(1.0/3.0) / r_d

    return interp1d(z_arr, D_V, kind='cubic', fill_value='extrapolate')

In [ ]:
def compute_chi_squared(sol, include_hubble=True, include_bao=True):
    """Compute chi2 against observational data."""
    if sol is None:
        return 1e6

    chi2 = 0.0
    n_data = 0

    z_arr = sol["z"]
    H_arr = sol["H"]

    from scipy.interpolate import interp1d

    if include_hubble and len(z_arr) > 2:
        H_interp = interp1d(z_arr, H_arr, kind='cubic', fill_value='extrapolate')
        for z_obs, H_obs, sigma_H in HUBBLE_DATA:
            if z_obs <= z_arr[-1] and z_obs >= z_arr[0]:
                H_model = float(H_interp(z_obs))
                chi2 += ((H_model - H_obs) / sigma_H)**2
                n_data += 1

    if include_bao and len(z_arr) > 2:
        try:
            DV_interp = compute_DV_from_solution(sol)
            for z_obs, DV_obs, sigma_DV, source in BAO_DATA:
                if z_obs <= z_arr[-1] and z_obs >= z_arr[0]:
                    DV_model = float(DV_interp(z_obs))
                    chi2 += ((DV_model - DV_obs) / sigma_DV)**2
                    n_data += 1
        except Exception:
            pass

    # Reduced chi-squared
    if n_data > 0:
        return chi2 / n_data
    return 1e6

============================================================================
PHANTOM CROSSING CHECK
============================================================================

In [ ]:
def check_phantom_crossing(sol):
    """Check if w(z) crosses -1 (phantom divide)."""
    if sol is None:
        return False, None, None

    w = sol["w"]
    z = sol["z"]

    # Look for sign changes in w + 1
    w_plus1 = w + 1.0
    crossings = []
    for i in range(len(w_plus1) - 1):
        if w_plus1[i] * w_plus1[i+1] < 0:
            # Linear interpolation to find crossing z
            z_cross = z[i] + (z[i+1] - z[i]) * abs(w_plus1[i]) / (abs(w_plus1[i]) + abs(w_plus1[i+1]))
            crossings.append(z_cross)

    has_crossing = len(crossings) > 0

    # Check w0 and w_a
    # w(z) ≈ w0 + w_a · z/(1+z) — CPL parameterization
    # w0 = w(z=0)
    w0 = float(np.interp(0.0, z, w)) if z[0] <= 0.01 else None

    return has_crossing, crossings, w0

============================================================================
OPTIMIZATION LOOP
============================================================================

In [ ]:
def optimize_potential(potential, chi_init_range=(0.5, 3.0),
                       chi_dot_init_range=(-1.0, 1.0),
                       n_de_iter=200, seed=2828):
    """Use differential evolution to find best parameters for a potential."""

    def objective(x):
        # x = [*potential_params, chi_init, chi_dot_init]
        pot_params = x[:potential.n_params]
        chi_init = x[potential.n_params]
        chi_dot_init = x[potential.n_params + 1]

        try:
            sol = solve_kg_friedmann(potential, pot_params,
                                     chi_init=chi_init,
                                     chi_dot_init=chi_dot_init,
                                     z_range=(0, 2.5))
            if sol is None:
                return 1e6

            chi2 = compute_chi_squared(sol)

            # Penalty: prefer phantom crossing
            has_crossing, _, w0 = check_phantom_crossing(sol)
            if not has_crossing:
                chi2 += 10.0  # soft penalty

            # Penalty: w0 should be near -1.28
            if w0 is not None:
                chi2 += 2.0 * (w0 - (-1.28))**2

            return chi2
        except Exception:
            return 1e6

    # Bounds: potential params + chi_init + chi_dot_init
    bounds = list(potential.bounds) + [chi_init_range, chi_dot_init_range]

    try:
        result = differential_evolution(objective, bounds, seed=seed,
                                         maxiter=n_de_iter, tol=1e-4,
                                         popsize=15, mutation=(0.5, 1.5),
                                         recombination=0.9)

        best_params = result.x[:potential.n_params]
        best_chi_init = result.x[potential.n_params]
        best_chi_dot_init = result.x[potential.n_params + 1]
        best_chi2 = result.fun

        # Get full solution at best params
        best_sol = solve_kg_friedmann(potential, best_params,
                                      chi_init=best_chi_init,
                                      chi_dot_init=best_chi_dot_init,
                                      z_range=(0, 2.5))

        has_crossing, crossings, w0 = check_phantom_crossing(best_sol)

        return {
            "potential": potential.name,
            "params": {n: float(v) for n, v in zip(potential.param_names, best_params)},
            "chi_init": float(best_chi_init),
            "chi_dot_init": float(best_chi_dot_init),
            "chi2_reduced": float(best_chi2),
            "phantom_crossing": has_crossing,
            "crossing_redshifts": [float(z) for z in crossings] if crossings else [],
            "w0": float(w0) if w0 is not None else None,
            "converged": result.success,
            "n_evaluations": result.nfev,
            "solution": best_sol,
        }
    except Exception as e:
        return {
            "potential": potential.name,
            "chi2_reduced": 1e6,
            "error": str(e),
            "phantom_crossing": False,
            "solution": None,
        }

============================================================================
EFFECTIVE w(z) PARAMETERIZATION (BASELINE — the chi2=1.54 result)
============================================================================

In [ ]:
def effective_w_model():
    """The effective parameterization that already works (chi2=1.54).
    w(z) = w0 + w_a·z/(1+z), CPL parameterization.
    This is the TARGET for the V(chi) search."""

    w0 = -1.28
    wa = 0.70

    # Build H(z) from CPL parameterization
    z_arr = np.linspace(0, 2.5, 500)
    H_arr = np.zeros_like(z_arr)

    Omega_de0 = 1.0 - Omega_m0 - Omega_r0

    for i, z in enumerate(z_arr):
        # Dark energy density evolution with CPL
        # ρ_de(z) = ρ_de0 · (1+z)^(3(1+w0+wa)) · exp(-3wa·z/(1+z))
        rho_de_ratio = (1+z)**(3*(1 + w0 + wa)) * np.exp(-3*wa*z/(1+z))

        H2 = H0_FIDUCIAL**2 * (Omega_m0*(1+z)**3 + Omega_r0*(1+z)**4 + Omega_de0*rho_de_ratio)
        H_arr[i] = np.sqrt(max(H2, 0))

    w_arr = w0 + wa * z_arr / (1 + z_arr)

    return {
        "z": z_arr,
        "H": H_arr,
        "w": w_arr,
        "chi": np.ones_like(z_arr),  # placeholder
        "rho_chi": np.ones_like(z_arr),
        "a": 1.0 / (1.0 + z_arr),
    }

============================================================================
MAIN EXECUTION
============================================================================

In [ ]:
def main():
    print("=" * 70)
    print("CHI-FIELD POTENTIAL SEARCH -- V(chi) FUNCTIONAL FORM SURVEY")
    print("=" * 70)
    print(f"Started: {datetime.now(timezone.utc).isoformat()}")
    print()

    # 1. Baseline: effective parameterization
    print("--- BASELINE: Effective w(z) parameterization ---")
    eff_sol = effective_w_model()
    eff_chi2 = compute_chi_squared(eff_sol)
    print(f"  Effective model chi2_reduced = {eff_chi2:.3f}")
    print(f"  w0 = -1.28, w_a = +0.70 (CPL)")
    print(f"  This is the TARGET. V(chi) must match or beat this.\n")

    # 2. Survey all potentials
    potentials = [
        OriginalHybrid(),
        DoubleExponential(),
        TrackerBarrier(),
        AlbrechtSkordis(),
        PNGB(),
        NonMinimalCoupled(),
        CoupledQuintessence(),
        ChiFreezeThaw(),
    ]

    all_results = []

    for pot in potentials:
        print(f"\n{'='*60}")
        print(f"TESTING: {pot.name}")
        print(f"  Parameters: {pot.param_names}")
        print(f"  Bounds: {pot.bounds}")
        print(f"  Optimizing (differential evolution, ~200 iterations)...")

        result = optimize_potential(pot, n_de_iter=200)

        chi2 = result.get("chi2_reduced", 1e6)
        crossing = result.get("phantom_crossing", False)
        w0 = result.get("w0", None)

        print(f"  chi2_reduced = {chi2:.3f}")
        print(f"  Phantom crossing: {crossing}")
        if w0 is not None:
            print(f"  w0 = {w0:.3f}")
        if "params" in result:
            for pname, pval in result["params"].items():
                print(f"  {pname} = {pval:.6f}")
        if "error" in result:
            print(f"  ERROR: {result['error']}")

        # Verdict
        if chi2 < 2.0 and crossing:
            verdict = "WINNER"
        elif chi2 < 5.0:
            verdict = "VIABLE"
        elif chi2 < 20.0:
            verdict = "MARGINAL"
        else:
            verdict = "REJECTED"

        print(f"  VERDICT: {verdict}")
        result["verdict"] = verdict

        # Don't serialize the full solution arrays
        result_clean = {k: v for k, v in result.items() if k != "solution"}
        all_results.append(result_clean)

    # 3. Summary table
    print("\n\n" + "=" * 70)
    print("SUMMARY TABLE — SORTED BY chi2")
    print("=" * 70)
    all_results.sort(key=lambda x: x.get("chi2_reduced", 1e6))

    print(f"{'Potential':<40} {'chi2':>8} {'w0':>8} {'Crossing':>10} {'Verdict':>10}")
    print("-" * 80)
    for r in all_results:
        name = r["potential"][:38]
        chi2 = r.get("chi2_reduced", 1e6)
        w0 = r.get("w0", None)
        crossing = r.get("phantom_crossing", False)
        verdict = r.get("verdict", "?")
        w0_str = f"{w0:.3f}" if w0 is not None else "N/A"
        print(f"  {name:<38} {chi2:>8.3f} {w0_str:>8} {'YES' if crossing else 'NO':>10} {verdict:>10}")

    print(f"\n  Effective model baseline: chi2 = {eff_chi2:.3f}")

    # 4. Best candidate analysis
    best = all_results[0]
    print(f"\n{'='*70}")
    print(f"BEST CANDIDATE: {best['potential']}")
    print(f"{'='*70}")
    print(f"  chi2_reduced = {best.get('chi2_reduced', 'N/A')}")
    print(f"  w0 = {best.get('w0', 'N/A')}")
    print(f"  Phantom crossing: {best.get('phantom_crossing', False)}")
    if "params" in best:
        print(f"  Parameters:")
        for pname, pval in best["params"].items():
            print(f"    {pname} = {pval}")

    if best.get("chi2_reduced", 1e6) > 2.0:
        print("\n  HONEST ASSESSMENT: No single potential achieved chi2 < 2.0")
        print("  from the raw Klein-Gordon ODE. This is a known difficulty.")
        print("  Possible reasons:")
        print("    1. Standard quintessence (canonical scalar) CANNOT cross w=-1")
        print("       without ghost instability (Vikman 2005, Caldwell & Doran 2005)")
        print("    2. Phantom crossing REQUIRES either:")
        print("       a) Non-canonical kinetic term (k-essence)")
        print("       b) Multiple coupled fields")
        print("       c) Non-minimal coupling to gravity (ξchi2R — which chi-field HAS)")
        print("    3. The effective parameterization chi2=1.54 works BECAUSE it")
        print("       parameterizes w(z) directly, bypassing the no-go theorem")
        print("  The chi-field action DOES have ξchi2R coupling — a full treatment")
        print("  requires solving the modified Einstein equations, not just")
        print("  Klein-Gordon in a fixed FLRW background.")

    # 5. Save results
    output = {
        "run_timestamp": datetime.now(timezone.utc).isoformat(),
        "baseline_chi2": round(eff_chi2, 4),
        "results": all_results,
        "best_candidate": best["potential"],
        "best_chi2": best.get("chi2_reduced", None),
        "recommendation": (
            "V(chi) raw ODE requires non-minimal coupling treatment. "
            "Standard quintessence cannot phantom-cross. "
            "The ξchi2R term in the chi-field action is the mechanism. "
            "Next step: solve the FULL modified Einstein + KG system with ξ≠0."
        )
    }

    output_dir = os.path.dirname(os.path.abspath(__file__))
    output_path = os.path.join(output_dir, "chi_field_potential_results.json")
    with open(output_path, "w") as f:
        json.dump(output, f, indent=2, default=str)
    print(f"\nResults saved to: {output_path}")

    print("\n" + "=" * 70)
    print("SEARCH COMPLETE")
    print("=" * 70)

    return output

In [ ]:
if __name__ == "__main__":
    main()